# Example 12.3 / 1: Grow a Cut Family Between Solves

Companion notebook for **12.3 Cuts and Subtour Elimination** in the AMPL lab Documentation, section "Grow a Cut Family Between Solves". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Grow a Cut Family Between Solves

The separator, rather than the solver's success status alone, decides whether the loop may stop.

The stopping test is structural: stop only when the selected incumbent (the best integer solution found so far) contains one tour (or the required number of depot routes). “The solver returned solved” means the current relaxation was solved; it does not mean all missing cuts are satisfied.

The model below declares the cut family `LeaveCut` over `1..nCuts` before any cut exists. After
separation, the run updates `nCuts` and `CUT_NODES` and solves again; AMPL instantiates the new
members as the set grows. The separator follows
selected arcs from the depot; when it reaches fewer than all nodes, the reached set becomes the
next `CUT_NODES` member. The run assigns `nCuts` by item assignment and an indexed set member with
`ampl.set["CUT_NODES"][k] = [...]`, and it evaluates every cut again from `to_dict()` at the end:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set NODES ordered;                      # depot first, then customers
set ARCS within {NODES, NODES};         # allowed directed arcs
param cost {ARCS} >= 0;                 # travel cost per arc
var Use {ARCS} binary;                  # 1 if the tour uses arc (i,j)

minimize TravelCost: sum {(i,j) in ARCS} cost[i,j] * Use[i,j];  # total travel cost
subject to Leave {i in NODES}: sum {(i,j) in ARCS} Use[i,j] = 1;  # one departure from every node
subject to Enter {j in NODES}: sum {(i,j) in ARCS} Use[i,j] = 1;  # one arrival at every node

param nCuts integer >= 0 default 0;     # number of cuts generated so far
set CUT_NODES {1..nCuts} within NODES;  # node set S of each cut
subject to LeaveCut {k in 1..nCuts}:    # at least one selected arc leaves each S
    sum {i in CUT_NODES[k], j in NODES diff CUT_NODES[k]: (i,j) in ARCS}
        Use[i,j] >= 1;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
nodes = ["depot", "c1", "c2", "c3", "c4"]
travel = {
    "depot": [0, 2, 2, 9, 9],
    "c1":    [2, 0, 1, 8, 9],
    "c2":    [2, 2, 0, 9, 8],
    "c3":    [9, 8, 9, 0, 1],
    "c4":    [9, 9, 7, 1, 0],
}
ampl.set["NODES"] = nodes
ampl.set["ARCS"] = [(i, j) for i in nodes for j in nodes if i != j]
ampl.param["cost"] = {(i, j): travel[i][k]
                      for i in nodes for k, j in enumerate(nodes) if i != j}

In [5]:
# run.py
def reached_from_depot(used):
    reached = {"depot"}
    while True:
        grown = reached | {j for (i, j) in used if i in reached}
        if grown == reached:
            return reached
        reached = grown

cuts = []
while True:
    ampl.solve(solver="highs", verbose=False)
    assert ampl.solve_result == "solved", ampl.solve_result
    used = [arc for arc, v in ampl.var["Use"].to_dict().items() if v > 0.5]
    reached = reached_from_depot(used)
    print(f"cuts {len(cuts)}: cost {ampl.obj['TravelCost'].value():g},"
          f" reached {sorted(reached)}")
    if len(reached) == len(nodes):
        break
    assert 0 < len(reached) < len(nodes)    # never an empty or full cut set
    cuts.append(sorted(reached))
    ampl.param["nCuts"] = len(cuts)
    ampl.set["CUT_NODES"][len(cuts)] = cuts[-1]

for k, members in enumerate(cuts, start=1):
    lhs = sum(1 for (i, j) in used if i in members and j not in members)
    print(f"LeaveCut[{k}] {members}: left-hand side {lhs} >= 1")
print("tour arcs:", used)

cuts 0: cost 7, reached ['c1', 'c2', 'depot']
cuts 1: cost 20, reached ['c1', 'c2', 'c3', 'c4', 'depot']
LeaveCut[1] ['c1', 'c2', 'depot']: left-hand side 1 >= 1
tour arcs: [('c1', 'c3'), ('c2', 'depot'), ('c3', 'c4'), ('c4', 'c2'), ('depot', 'c1')]


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`. `setof` collects the successors of the
reached nodes until the set stops growing:

In [6]:
%%writefile data.dat
set NODES := depot c1 c2 c3 c4;
set ARCS:  depot c1 c2 c3 c4 :=
    depot    -    +  +  +  +
    c1       +    -  +  +  +
    c2       +    +  -  +  +
    c3       +    +  +  -  +
    c4       +    +  +  +  - ;
param cost:  depot c1 c2 c3 c4 :=
    depot    .    2  2  9  9
    c1       2    .  1  8  9
    c2       2    2  .  9  8
    c3       9    8  9  .  1
    c4       9    9  7  1  . ;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs, solver_msg 0;
set REACHED within NODES;  # nodes the current tour reaches from the depot
param size_before;         # card(REACHED) before the last growth step
repeat {
    solve;
    let REACHED := {first(NODES)};
    repeat {
        let size_before := card(REACHED);
        let REACHED := REACHED union
            setof {(i,j) in ARCS: i in REACHED and Use[i,j] > 0.5} j;
    } until card(REACHED) = size_before;
    printf "cuts %d: cost %g, reached %d of %d nodes\n",
        nCuts, TravelCost, card(REACHED), card(NODES);
    if card(REACHED) = card(NODES) then break;
    let nCuts := nCuts + 1;
    let CUT_NODES[nCuts] := REACHED;
};
display CUT_NODES;
display LeaveCut.body;

cuts 0: cost 7, reached 3 of 5 nodes
cuts 1: cost 20, reached 5 of 5 nodes
set CUT_NODES[1] := depot c1 c2;

LeaveCut.body [*] :=
1  1
;



### Another way in amplpy

The first solve returns the cycle cover of cost 7; one cut on {depot, c1, c2} is enough, and the
second solve returns the tour of cost 20. The final check evaluates the cut at the final tour:
one arc, c1-c3, leaves the cut set.

A cut set must be a nonempty proper subset. The same model with a full or an empty set turns
`LeaveCut` into $0 \ge 1$, so check the set's size before adding a cut:

In [9]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [10]:
# data.py
nodes = ["depot", "c1", "c2", "c3", "c4"]
travel = {
    "depot": [0, 2, 2, 9, 9],
    "c1":    [2, 0, 1, 8, 9],
    "c2":    [2, 2, 0, 9, 8],
    "c3":    [9, 8, 9, 0, 1],
    "c4":    [9, 9, 7, 1, 0],
}
ampl.set["NODES"] = nodes
ampl.set["ARCS"] = [(i, j) for i in nodes for j in nodes if i != j]
ampl.param["cost"] = {(i, j): travel[i][k]
                      for i in nodes for k, j in enumerate(nodes) if i != j}

In [11]:
# run.py
for members in [nodes, []]:
    ampl.param["nCuts"] = 1
    ampl.set["CUT_NODES"][1] = members
    ampl.solve(solver="highs", verbose=False)
    print(len(members), "nodes in the cut set:", ampl.solve_result)

5 nodes in the cut set: infeasible
0 nodes in the cut set: infeasible
